# 노트북 6. Streaming 응답

> Phase 2 — 제어: 모델 동작을 원하는 대로 다루기

사용자는 빈 화면에서 3초 기다리는 것보다, 글자가 하나씩 나오는 것을 훨씬 빠르다고 느낍니다.
이 노트북에서는 스트리밍의 동작 원리를 이해하고, 직접 구현하는 방법을 배웁니다.

**학습 목표**
- 스트리밍과 비스트리밍의 동작 차이를 설명할 수 있다
- google-genai와 LangChain에서 스트리밍을 구현할 수 있다
- 청크(Chunk)의 구조를 분석하고 텍스트를 누적할 수 있다
- TTFT(Time To First Token)의 개념을 이해하고 직접 측정할 수 있다
- LCEL 체인에서 스트리밍을 적용할 수 있다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | 스트리밍 원리 + google-genai/LangChain 구현 + TTFT 측정 | 읽고 실행 |
| Part 2 — 실습 | 스트리밍 구현 + 청크 분석 + 성능 측정 | 직접 작성 |
| Part 3 — 챌린지 | 스트리밍 vs 비스트리밍 성능 비교 실험 | 강사와 함께 |

In [1]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai

import os
import time
from google import genai

# API 키 설정 (Colab 환경 기준)
# 왼쪽 키 아이콘 → GEMINI_API_KEY 등록 후 실행
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL = "gemini-2.5-flash"
print("환경 설정 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 926.6 kB/s eta 0:00:00
환경 설정 완료


---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 스트리밍이란 무엇인가?

LLM API 호출에는 두 가지 방식이 있습니다.

**비스트리밍 (Non-streaming)**
- 모델이 전체 응답을 생성한 후 한 번에 반환
- 응답이 길수록 사용자가 빈 화면을 오래 바라봐야 함

**스트리밍 (Streaming)**
- 토큰이 생성되는 즉시 청크(chunk) 단위로 반환
- 첫 토큰이 나오는 순간부터 화면에 텍스트가 표시됨

```
비스트리밍:
  요청 ─────────────[전체 생성 대기]─────────────> 응답 한 번에 수신

스트리밍:
  요청 ──[TTFT]──> 청크1 > 청크2 > 청크3 > ... > 완료
                   ↑ 여기서부터 화면에 표시 시작
```

총 생성 시간은 비슷하지만, 사용자가 **체감하는 대기 시간**은 스트리밍이 훨씬 짧습니다.

아래 코드는 비스트리밍 호출의 대기 시간을 측정합니다.

In [4]:
# 비스트리밍 호출 — 전체 응답을 기다린 후 출력
prompt = "파이썬의 주요 특징 5가지를 설명해주세요."

start = time.time()
response = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    # thinking_budget = 0
    config={"thinking_config": {"thinking_budget": 0}},
)
elapsed = time.time() - start

print(f"[비스트리밍] 총 대기 시간: {elapsed:.2f}초")
print(f"출력 토큰: {response.usage_metadata.candidates_token_count}")
print(f"\n{response.text[:200]}...")

[비스트리밍] 총 대기 시간: 6.41초
출력 토큰: 1263

## 파이썬의 주요 특징 5가지

파이썬은 배우기 쉽고 강력한 범용 프로그래밍 언어로, 다양한 분야에서 널리 사용되고 있습니다. 다음은 파이썬의 주요 특징 5가지입니다.

### 1. 배우기 쉽고 가독성이 높음 (Easy to Learn & High Readability)

* **간결한 문법:** 파이썬은 다른 프로그래밍 언어에 비해 문법이 매우 간결하고...


이제 같은 프롬프트를 스트리밍으로 호출합니다.
첫 번째 청크가 도착하는 시간(TTFT)과 전체 완료 시간을 비교해보세요.

스트리밍은 HTTP의 **Server-Sent Events(SSE)** 프로토콜을 사용합니다.
서버가 연결을 유지한 채로 데이터를 조각조각 보내주는 방식입니다.

In [8]:
# 스트리밍 호출 — 청크가 도착할 때마다 출력
start = time.time()
ttft = None
full_text = ""
chunk_count = 0

for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
):
    if ttft is None:
        ttft = time.time() - start
    full_text += chunk.text if chunk is not None else ""
    chunk_count += 1
    print(chunk.text, end="", flush=True)

total = time.time() - start
print(f"\n\n--- 측정 결과 ---")
print(f"TTFT (첫 청크 도착): {ttft:.2f}초")
print(f"Total (전체 완료): {total:.2f}초")
print(f"청크 수: {chunk_count}개")

## 파이썬의 주요 특징 5가지

파이썬은 배우기 쉽고 강력한 프로그래밍 언어로 다양한 분야에서 활용됩니다. 다음은 파이썬의 핵심적인 특징 5가지입니다.

---

### 1. 배우기 쉬운 문법 (Simple and Readable Syntax)

*   **직관적인 코드:** 파이썬은 자연어와 유사한 문법 구조를 가지고 있어 코드를 읽고 이해하기 쉽습니다. 다른 언어에 비해 훨씬 적은 코드로 동일한 기능을 구현할 수 있습니다.
*   **들여쓰기(Indentation):** 파이썬은 코드 블록을 구분하기 위해 중괄호(`{}`) 대신 들여쓰기를 사용합니다. 이는 코드를 시각적으로 깔끔하게 만들고 가독성을 높여줍니다.
*   **빠른 학습 곡선:** 이러한 특징 덕분에 프로그래밍 초보자도 빠르게 언어를 익히고 실제 애플리케이션 개발에 착수할 수 있습니다.

### 2. 다양한 라이브러리와 프레임워크 (Extensive Libraries and Frameworks)

*   **풍부한 생태계:** 파이썬은 방대한 양의 표준 라이브러리와 서드파티 라이브러리(패키지)를 제공합니다. 이들은 데이터 과학, 웹 개발, 인공지능, 자동화 등 거의 모든 분야를 포괄합니다.
*   **예시:**
    *   **웹 개발:** Django, Flask
    *   **데이터 분석 및 과학:** NumPy, Pandas, Matplotlib, SciPy
    *   **머신러닝 및 인공지능:** TensorFlow, Keras, PyTorch, scikit-learn
    *   **GUI 개발:** PyQt, Tkinter
    *   **자동화:** Selenium
*   **개발 시간 단축:** 이미 구현된 기능을 재사용함으로써 개발자는 핵심 로직에 집중하고 개발 시간을 크게 단축할 수 있습니다.

### 3. 인터프리터 언어 (Interpreted Language)

*   **실행 방식:** 파이썬 코드는 컴파일 과정 없이 인터프리터에 의해 한 줄씩 즉시 실행됩니

| 지표 | 비스트리밍 | 스트리밍 |
|------|----------|--------|
| 사용자가 첫 글자를 보는 시간 | 전체 생성 완료 후 | TTFT (보통 0.5~2초) |
| 전체 응답 완료 시간 | 거의 동일 | 거의 동일 |
| 네트워크 요청 수 | 1회 | 1회 (SSE 연결 유지) |
| 구현 복잡도 | 낮음 | 약간 높음 (청크 처리 필요) |
| UX 체감 | 느림 | 빠름 |

## 1.2 google-genai 스트리밍: generate_content_stream()

google-genai SDK에서 스트리밍은 `generate_content_stream()` 메서드를 사용합니다.
일반 호출의 `generate_content()`와 파라미터가 동일하며, 반환값만 이터레이터로 바뀝니다.

```python
# 비스트리밍
response = client.models.generate_content(model=MODEL, contents=prompt)

# 스트리밍 — 메서드 이름만 다름
for chunk in client.models.generate_content_stream(model=MODEL, contents=prompt):
    print(chunk.text, end="")
```

아래 코드는 각 청크의 도착 시점을 타임스탬프로 기록합니다.

In [9]:
# 청크별 타임스탬프 기록
prompt = "머신러닝과 딥러닝의 차이를 간단히 설명해줘"

start = time.time()
chunks_info = []

for i, chunk in enumerate(client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
)):
    elapsed = time.time() - start
    text_len = len(chunk.text)
    chunks_info.append((i, elapsed, text_len, chunk.text[:30]))

print(f"{'번호':>4} {'시간(초)':>8} {'글자수':>6}  내용(앞 30자)")
print("-" * 60)
for idx, t, length, preview in chunks_info:
    print(f"{idx:>4} {t:>8.3f} {length:>6}  {preview}")

print(f"\n총 {len(chunks_info)}개 청크, 총 시간: {chunks_info[-1][1]:.2f}초")

  번호    시간(초)    글자수  내용(앞 30자)
------------------------------------------------------------
   0    0.610      1  네
   1    0.724     33  , 간단하게 비유를 들어 설명해 드릴게요.

**머신러
   2    1.047     95  기계 학습) 전체 스펙트럼**

머신러닝은 컴퓨터에게 
   3    1.288     86   큰 개념이에요. 마치 사람에게 "공부시키는 방법" 전
   4    1.531     94  **

딥러닝은 이 "공부시키는 방법" 중에서도 **아
   5    1.826     82  "을 여러 층(깊게) 쌓아서 학습하는 방식**이에요.

   6    2.139     87  , 일식 등 모든 요리 포함)
*   **딥러닝:** 
   7    2.404    100  )

**핵심 차이:**

1.  **데이터 표현 학습
   8    2.683     95   때가 많아요. (예: "사진에서 고양이 귀와 수염을 
   9    2.968     89   고양이 사진을 엄청 많이 보여주면, 딥러닝 모델이 알
  10    3.186     96  의 가장 큰 장점이에요.

2.  **데이터 양:**

  11    3.431     86  .
    *   **딥러닝:** 엄청나게 **많은 양
  12    3.634     93  잡성:**
    *   **머신러닝 (일부 전통 방식
  13    3.881    100  닝:** 훨씬 더 **복잡하고 추상적인 패턴**을 학습
  14    4.091     86  약:**

*   **머신러닝**은 컴퓨터가 데이터로부
  15    4.364     91  , **인공신경망을 깊게 쌓아 데이터를 스스로 학습하고
  16    4.500     42   성과들(AlphaGo, 챗GPT, 자율주행 등)은 딥

총 17개 청크, 총 시간: 4.50초


## 1.3 청크(Chunk) 구조 분석

스트리밍의 각 청크는 `GenerateContentResponse` 객체와 동일한 구조를 가집니다.
다만 각 청크에는 전체 응답의 일부분만 담겨 있습니다.

주요 필드:
- `chunk.text`: 이 청크의 텍스트 내용 (편의 접근자)
- `chunk.candidates[0].content.parts`: Part 리스트
- `chunk.candidates[0].finish_reason`: 마지막 청크에서만 의미 있음
- `chunk.usage_metadata`: 마지막 청크에서 토큰 사용량 포함

아래 코드는 첫 번째 청크와 마지막 청크의 구조를 비교합니다.

In [10]:
# 청크 구조 분석 — 첫 번째와 마지막 비교
prompt = "클라우드 컴퓨팅이란?"
all_chunks = []

for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
):
    all_chunks.append(chunk)

print(f"총 청크 수: {len(all_chunks)}")

# 첫 번째 청크
first = all_chunks[0]
print(f"\n=== 첫 번째 청크 ===")
print(f"text: {first.text[:50]}")
print(f"finish_reason: {first.candidates[0].finish_reason}")
print(f"usage_metadata: {first.usage_metadata}")

# 마지막 청크
last = all_chunks[-1]
print(f"\n=== 마지막 청크 ===")
print(f"text: {last.text[:50]}")
print(f"finish_reason: {last.candidates[0].finish_reason}")
print(f"usage_metadata: {last.usage_metadata}")

총 청크 수: 22

=== 첫 번째 청크 ===
text: 클
finish_reason: None
usage_metadata: cache_tokens_details=None cached_content_token_count=None candidates_token_count=1 candidates_tokens_details=None prompt_token_count=8 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=8
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=9 traffic_type=None

=== 마지막 청크 ===
text: 은 현대 IT 환경의 핵심적인 부분이며, 개인부터 대기업에 이르기까지 거의 모든 산업에서 
finish_reason: FinishReason.STOP
usage_metadata: cache_tokens_details=None cached_content_token_count=None candidates_token_count=976 candidates_tokens_details=None prompt_token_count=8 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=8
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=984 traffic_type=None


### 텍스트 누적 패턴

스트리밍에서는 각 청크가 전체 응답의 일부이므로, 최종 텍스트를 얻으려면 청크를 이어붙여야 합니다.

```python
full_text = ""
for chunk in client.models.generate_content_stream(...):
    full_text += chunk.text
# full_text에 전체 응답이 누적됨
```

아래 코드는 누적 과정을 시각적으로 보여줍니다.

In [12]:
# 텍스트 누적 — 진행률 표시
prompt = "데이터 사이언스의 핵심 단계 3가지를 간단히 설명해줘"

full_text = ""
chunk_count = 0

for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
):
    full_text += chunk.text if chunk is not None else ""
    chunk_count += 1

print(f"청크 수: {chunk_count}")
print(f"누적된 전체 텍스트 길이: {len(full_text)}자")
print(f"\n--- 전체 응답 ---")
print(full_text)

청크 수: 7
누적된 전체 텍스트 길이: 485자

--- 전체 응답 ---
데이터 사이언스의 핵심 단계 3가지는 다음과 같습니다.

1.  **문제 정의 및 데이터 수집:** 어떤 질문에 답하고 싶은지, 어떤 문제를 해결하고 싶은지 명확히 정의합니다. 그리고 이 문제를 해결하는 데 필요한 데이터를 다양한 소스(데이터베이스, 웹사이트, 센서 등)에서 모으는 단계입니다.

2.  **데이터 탐색 및 전처리:** 수집된 데이터를 이해하고 분석하기 쉬운 형태로 만드는 단계입니다. 데이터의 특징을 파악하고, 누락된 값, 잘못된 값 등을 처리하며, 필요한 경우 데이터를 변환하거나 조합합니다. 이 과정에서 시각화 도구를 활용하여 데이터의 패턴과 이상치를 찾아낼 수 있습니다.

3.  **모델링 및 평가:** 깨끗하게 준비된 데이터를 사용하여 예측, 분류, 군집화 등 특정 목표를 달성할 수 있는 통계적 또는 머신러닝 모델을 구축합니다. 모델을 학습시킨 후, 이 모델이 얼마나 정확하고 효과적인지 평가하고, 필요에 따라 모델을 개선하는 단계입니다.


### 청크 크기의 불균일성

청크의 크기는 균일하지 않습니다.

이는 모델의 토큰 생성 속도와 네트워크 버퍼링에 따라 달라집니다.

아래 코드는 청크별 텍스트 길이를 시각적으로 표시합니다.

In [13]:
# 청크별 크기 시각화 (텍스트 막대)
prompt = "좋은 소프트웨어 엔지니어가 되려면 어떤 역량이 필요한지 설명해주세요."

chunk_sizes = []
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
):
    chunk_sizes.append(len(chunk.text))

# 막대 그래프 출력
max_size = max(chunk_sizes) if chunk_sizes else 1
print(f"총 {len(chunk_sizes)}개 청크\n")
for i, size in enumerate(chunk_sizes):
    bar = "#" * int(size / max_size * 40)
    print(f"청크 {i:>2}: {bar:<40} ({size}자)")

총 24개 청크

청크  0:                                          (1자)
청크  1: #############                            (39자)
청크  2: #################################        (101자)
청크  3: ###############################          (94자)
청크  4: ####################################     (110자)
청크  5: ###############################          (95자)
청크  6: #################################        (99자)
청크  7: ######################################## (119자)
청크  8: #################################        (99자)
청크  9: #############################            (89자)
청크 10: ###############################          (93자)
청크 11: ##################################       (103자)
청크 12: #################################        (99자)
청크 13: ##############################           (91자)
청크 14: ####################################     (109자)
청크 15: ####################################     (108자)
청크 16: ###############################          (95자)
청크 17: ##############################           (90자)
청크 18: ######

## 1.4 스트리밍 토큰 사용량

스트리밍에서도 토큰 사용량을 확인할 수 있습니다.
다만, `usage_metadata`는 **마지막 청크**에서만 의미 있는 값이 포함됩니다.

아래 코드는 마지막 청크에서 토큰 사용량을 추출합니다.

In [14]:
# 마지막 청크에서 토큰 사용량 확인
prompt = "양자컴퓨팅의 원리를 간단히 설명해줘"
last_chunk = None

for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
):
    last_chunk = chunk

usage = last_chunk.usage_metadata
print(f"입력 토큰: {usage.prompt_token_count}")
print(f"출력 토큰: {usage.candidates_token_count}")
print(f"총 토큰: {usage.total_token_count}")

입력 토큰: 14
출력 토큰: 495
총 토큰: 509


In [16]:
# 스트리밍 vs 비스트리밍 토큰 비교 — 같은 프롬프트
prompt = "양자컴퓨팅의 원리를 한 문장으로 설명해줘"

# 비스트리밍
resp_normal = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
)

# 스트리밍 (마지막 청크에서 추출)
last_chunk = None
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
):
    last_chunk = chunk

print(f"{'':>15} {'비스트리밍':>12} {'스트리밍':>12}")
print("-" * 42)
print(f"{'입력 토큰':>15} {resp_normal.usage_metadata.prompt_token_count:>12} {last_chunk.usage_metadata.prompt_token_count:>12}")
print(f"{'출력 토큰':>15} {resp_normal.usage_metadata.candidates_token_count:>12} {last_chunk.usage_metadata.candidates_token_count:>12}")
print(f"{'총 토큰':>15} {resp_normal.usage_metadata.total_token_count:>12} {last_chunk.usage_metadata.total_token_count:>12}")
print("\n(입력 토큰은 동일, 출력 토큰은 생성 결과에 따라 다를 수 있음)")

                       비스트리밍         스트리밍
------------------------------------------
          입력 토큰           16           16
          출력 토큰           38           37
           총 토큰           54           53

(입력 토큰은 동일, 출력 토큰은 생성 결과에 따라 다를 수 있음)


> 스트리밍 토큰 주의사항
> - 스트리밍이든 비스트리밍이든 **소비되는 토큰 수와 비용은 동일**합니다
> - 스트리밍은 UX 개선 도구이지, 비용 절감 도구가 아닙니다
> - 토큰 사용량은 반드시 마지막 청크에서 확인해야 합니다

## 1.5 LangChain 스트리밍: .stream()

LangChain의 `ChatGoogleGenerativeAI`는 `.stream()` 메서드로 스트리밍을 지원합니다.
반환되는 각 청크는 `AIMessageChunk` 타입입니다.

```python
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 비스트리밍
response = llm.invoke("질문")  # → AIMessage

# 스트리밍
for chunk in llm.stream("질문"):  # → AIMessageChunk
    print(chunk.content, end="")
```

아래 코드는 LangChain 스트리밍을 실행합니다.

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY,
    thinking_budget = 16
)

# LangChain 스트리밍
prompt = "블록체인 기술의 핵심 원리를 간단히 설명해줘"

for chunk in llm.stream(prompt):
    print(chunk.content, end="", flush=True)

print()  # 줄바꿈

블록체인 기술의 핵심 원리는 **'분산된 공개 장부'**라고 생각하시면 간단합니다.

조금 더 자세히 설명하면 다음과 같습니다.

1.  **블록 (Block):** 거래(트랜잭션) 기록들이 모여 하나의 묶음(블록)을 이룹니다. 이 블록에는 이전 블록의 정보(해시값)가 포함되어 있습니다.

2.  **체인 (Chain):** 각 블록은 이전 블록과 암호학적으로 연결되어 마치 사슬처럼 이어집니다. 이 연결 고리 때문에 한 번 기록된 정보는 위변조가 거의 불가능합니다. 중간의 블록을 수정하려면 그 이후의 모든 블록을 다 바꿔야 하는데, 이는 사실상 불가능합니다.

3.  **분산 원장 (Distributed Ledger):** 이 장부(블록체인)는 특정 중앙 기관이 관리하는 것이 아니라, 네트워크에 참여하는 모든 사용자(노드)들이 똑같은 사본을 가지고 공유합니다.

4.  **합의 (Consensus):** 새로운 블록이 생성되어 장부에 추가되려면, 네트워크 참여자들의 과반수 이상이 '이 블록은 유효하다'고 동의(합의)해야 합니다. 가장 대표적인 합의 방식은 '작업 증명(Proof of Work, PoW)'입니다.

**간단히 요약하면:**

*   **거래 기록들을 묶어 '블록'으로 만들고,**
*   **이 블록들을 서로 연결하여 '체인'을 이루며,**
*   **이 '블록체인 장부'를 모든 참여자가 '공유'하고,**
*   **새로운 기록은 '다수의 동의'를 거쳐 추가됩니다.**

이러한 특성 덕분에 블록체인은 투명하고, 위변조가 어렵고, 특정 중앙 기관 없이도 신뢰를 구축할 수 있는 기술이 됩니다.


## 1.6 AIMessageChunk 구조 분석

`AIMessageChunk`는 LangChain의 스트리밍 전용 메시지 타입입니다.
일반 `AIMessage`와 비슷한 구조이지만, 전체 응답의 일부분만 담고 있습니다.

주요 필드:
- `chunk.content`: 이 청크의 텍스트
- `chunk.id`: 메시지 고유 ID (모든 청크가 동일)
- `chunk.response_metadata`: 응답 메타데이터 (마지막 청크에서 채워짐)
- `chunk.usage_metadata`: 토큰 사용량 (마지막 청크에서 채워짐)

아래 코드는 청크의 필드를 분석합니다.

In [19]:
# AIMessageChunk 필드 분석
prompt = "API란 무엇인가?"
all_chunks = []

for chunk in llm.stream(prompt):
    all_chunks.append(chunk)

print(f"총 청크 수: {len(all_chunks)}")

# 첫 번째 청크
first = all_chunks[0]
print(f"\n=== 첫 번째 청크 ===")
print(f"type: {type(first).__name__}")
print(f"content: {first.content[:50]}")
print(f"id: {first.id}")
print(f"usage_metadata: {first.usage_metadata}")

# 마지막 청크
last = all_chunks[-1]
print(f"\n=== 마지막 청크 ===")
print(f"content: {last.content[:50]}")
print(f"usage_metadata: {last.usage_metadata}")
print(f"response_metadata: {last.response_metadata}")

총 청크 수: 25

=== 첫 번째 청크 ===
type: AIMessageChunk
content: API(Application Programming
id: lc_run--019c7459-ae2c-7873-bbe7-52002ece9b2b
usage_metadata: {'input_tokens': 6, 'output_tokens': 18, 'total_tokens': 24, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 14}}

=== 마지막 청크 ===
content: 
usage_metadata: None
response_metadata: {}


In [24]:
# 스트리밍 청크 상세 분석
for i, chunk in enumerate(all_chunks):
    content_preview = chunk.content[:30] if chunk.content else "(빈 content)"
    reasoning = chunk.usage_metadata.get('output_token_details', {}).get('reasoning', 0) if chunk.usage_metadata else "-"
    output_tokens = chunk.usage_metadata.get('output_tokens', 0) if chunk.usage_metadata else "-"
    input_tokens = chunk.usage_metadata.get('input_tokens', 0) if chunk.usage_metadata else "-"
    finish = chunk.response_metadata.get('finish_reason', '')

    print(f"[청크 {i+1:2d}]  input: {str(input_tokens):>3s} | output: {str(output_tokens):>3s} | reasoning: {str(reasoning):>3s} | finish: {finish}")


[청크  1]  input:   6 | output:  18 | reasoning:  14 | finish: 
[청크  2]  input:   0 | output:  49 | reasoning:   0 | finish: 
[청크  3]  input:   0 | output:  51 | reasoning:   0 | finish: 
[청크  4]  input:   0 | output:  49 | reasoning:   0 | finish: 
[청크  5]  input:   0 | output:  49 | reasoning:   0 | finish: 
[청크  6]  input:   0 | output:  48 | reasoning:   0 | finish: 
[청크  7]  input:   0 | output:  49 | reasoning:   0 | finish: 
[청크  8]  input:   0 | output:  48 | reasoning:   0 | finish: 
[청크  9]  input:   0 | output:  51 | reasoning:   0 | finish: 
[청크 10]  input:   0 | output:  48 | reasoning:   0 | finish: 
[청크 11]  input:   0 | output:  50 | reasoning:   0 | finish: 
[청크 12]  input:   0 | output:  48 | reasoning:   0 | finish: 
[청크 13]  input:   0 | output:  53 | reasoning:   0 | finish: 
[청크 14]  input:   0 | output:  48 | reasoning:   0 | finish: 
[청크 15]  input:   0 | output:  50 | reasoning:   0 | finish: 
[청크 16]  input:   0 | output:  49 | reasoning:   0 | finish: 
[청크 17] 

## 1.7 청크 병합 (+ 연산자)

`AIMessageChunk`는 `+` 연산자를 지원합니다.
청크를 하나씩 더하면 최종적으로 완전한 `AIMessageChunk`가 됩니다.

```python
full = None
for chunk in llm.stream("질문"):
    full = chunk if full is None else full + chunk
# full.content에 전체 텍스트
# full.usage_metadata에 토큰 사용량
```

이 패턴은 스트리밍 중 실시간 출력을 하면서, 동시에 전체 결과도 보존해야 할 때 유용합니다.

In [25]:
# 청크 병합 — + 연산자
prompt = "REST API의 HTTP 메서드 4가지를 설명해줘"

full = None
chunk_count = 0

for chunk in llm.stream(prompt):
    full = chunk if full is None else full + chunk
    chunk_count += 1
    print(chunk.content, end="", flush=True)

print(f"\n\n--- 병합 결과 ---")
print(f"청크 수: {chunk_count}")
print(f"전체 텍스트 길이: {len(full.content)}자")
print(f"토큰 사용량: {full.usage_metadata}")

네, REST API에서 가장 일반적으로 사용되는 HTTP 메서드 4가지 (CRUD 작업과 밀접하게 관련됨)를 설명해 드리겠습니다.

---

### REST API의 주요 HTTP 메서드 4가지

RESTful API는 웹의 기본 원칙인 HTTP 메서드를 리소스에 대한 "행동"으로 사용하여 통신합니다. 이 4가지 메서드는 대부분의 CRUD(Create, Read, Update, Delete) 작업을 나타냅니다.

---

#### 1. `GET` (조회/읽기 - Read)

*   **목적:** 특정 리소스 또는 리소스 컬렉션을 **조회(읽기)**할 때 사용합니다. 서버로부터 데이터를 요청하고 응답으로 받습니다.
*   **특징:**
    *   **안전(Safe):** 서버의 상태를 변경하지 않습니다. 여러 번 호출해도 항상 동일한 결과를 반환하며, 데이터에 부작용을 일으키지 않습니다.
    *   **멱등(Idempotent):** 여러 번 호출해도 한 번 호출한 것과 동일한 효과를 가집니다 (서버 상태 변경 측면에서).
    *   요청 본문(body)에 데이터를 포함하지 않습니다. 필요한 데이터는 URL의 쿼리 파라미터로 전달합니다.
    *   응답은 보통 200 OK 상태 코드와 함께 요청된 리소스 데이터를 포함합니다.
*   **예시:**
    *   `GET /users` : 모든 사용자 목록을 조회합니다.
    *   `GET /users/123` : ID가 123인 특정 사용자 정보를 조회합니다.
    *   `GET /products?category=electronics&limit=10` : 'electronics' 카테고리의 제품 10개를 조회합니다.

---

#### 2. `POST` (생성 - Create)

*   **목적:** 새로운 리소스를 **생성**할 때 사용합니다. 클라이언트가 서버에 데이터를 전송하여 새로운 리소스를 만들도록 요청합니다.
*   **특징:**
    *   **안전하지 않음(Not Safe)

> google-genai vs LangChain 스트리밍 비교
>
> | 항목 | google-genai | LangChain |
> |------|-------------|----------|
> | 메서드 | `generate_content_stream()` | `.stream()` |
> | 청크 타입 | `GenerateContentResponse` | `AIMessageChunk` |
> | 텍스트 접근 | `chunk.text` | `chunk.content` |
> | 토큰 사용량 | `chunk.usage_metadata` (마지막) | `chunk.usage_metadata` (마지막) |
> | 청크 병합 | 수동 (`+=` 문자열) | `+` 연산자 지원 |
> | 비동기 | `client.aio.models.generate_content_stream()` | `.astream()` |

## 1.8 TTFT (Time To First Token)

**TTFT(Time To First Token)**는 요청을 보낸 시점부터 첫 번째 토큰(청크)이 도착하는 시점까지의 시간입니다.

TTFT는 스트리밍의 가장 중요한 성능 지표입니다:
- 사용자가 "응답이 시작됐다"고 느끼는 시점을 결정
- 모델 크기, 프롬프트 길이, 서버 부하에 영향을 받음
- 일반적으로 0.5~3초 범위

아래 코드는 TTFT를 정밀하게 측정합니다.

In [26]:
# TTFT 측정 — google-genai
prompt = "인공지능 윤리에 대해 간단히 설명해줘"

start = time.time()
first_chunk_text = ""

for i, chunk in enumerate(client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
)):
    if i == 0:
        ttft = time.time() - start
        first_chunk_text = chunk.text[:50]

total = time.time() - start

print(f"TTFT: {ttft:.3f}초")
print(f"Total: {total:.3f}초")
print(f"TTFT 비율: {ttft/total*100:.1f}%")
print(f"첫 청크 내용: {first_chunk_text}")

TTFT: 0.704초
Total: 2.553초
TTFT 비율: 27.6%
첫 청크 내용: 인공지능


### TTFT 측정 — LangChain 버전

LangChain에서도 동일한 방식으로 TTFT를 측정할 수 있습니다.

In [27]:
# TTFT 측정 — LangChain
prompt = "인공지능 윤리에 대해 간단히 설명해줘"

start = time.time()

for i, chunk in enumerate(llm.stream(prompt)):
    if i == 0:
        ttft = time.time() - start
        first_content = chunk.content[:50]

total = time.time() - start

print(f"TTFT: {ttft:.3f}초")
print(f"Total: {total:.3f}초")
print(f"첫 청크 내용: {first_content}")

TTFT: 0.716초
Total: 2.920초
첫 청크 내용: 인공지능(


> TTFT가 중요한 이유 (UX 관점)
> - 연구에 따르면 사용자는 **1초 이내** 응답이 시작되면 "빠르다"고 느낍니다
> - 3초 이상 아무 반응이 없으면 "멈췄나?" 하고 불안해합니다
> - 스트리밍은 총 응답 시간을 줄이지 않지만, TTFT를 낮춰 **체감 속도를 크게 개선**합니다
> - ChatGPT, Gemini 등 모든 주요 챗봇 서비스가 스트리밍을 기본으로 사용하는 이유입니다

## 1.9 TTFT vs Total Time 비교

TTFT와 Total Time은 서로 다른 지표입니다.

- **TTFT**: 사용자가 첫 글자를 보는 시간 → UX 지표
- **Total Time**: 전체 응답이 완료되는 시간 → 처리량 지표

아래 코드는 같은 프롬프트에 대해 비스트리밍의 Total Time과 스트리밍의 TTFT를 비교합니다.

In [28]:
# 비스트리밍 Total Time vs 스트리밍 TTFT 비교
prompt = "자연어 처리(NLP)의 주요 기술 5가지를 설명해주세요."

# 비스트리밍
start = time.time()
resp = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
)
non_stream_total = time.time() - start

# 스트리밍
start = time.time()
stream_ttft = None
for i, chunk in enumerate(client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
)):
    if i == 0:
        stream_ttft = time.time() - start
stream_total = time.time() - start

print(f"{'비스트리밍 Total':>20}: {non_stream_total:.3f}초 ← 사용자가 첫 글자를 보는 시간")
print(f"{'스트리밍 TTFT':>20}: {stream_ttft:.3f}초 ← 사용자가 첫 글자를 보는 시간")
print(f"{'스트리밍 Total':>20}: {stream_total:.3f}초 ← 전체 응답 완료 시간")
print(f"\nUX 개선: 사용자가 {non_stream_total - stream_ttft:.2f}초 더 빨리 응답을 봅니다")

         비스트리밍 Total: 9.904초 ← 사용자가 첫 글자를 보는 시간
           스트리밍 TTFT: 0.505초 ← 사용자가 첫 글자를 보는 시간
          스트리밍 Total: 7.163초 ← 전체 응답 완료 시간

UX 개선: 사용자가 9.40초 더 빨리 응답을 봅니다


### 프롬프트 길이와 TTFT

프롬프트가 길수록 TTFT가 증가합니다.
모델이 입력을 처리하는 데 더 많은 시간이 걸리기 때문입니다.

아래 코드는 짧은 프롬프트와 긴 프롬프트의 TTFT를 비교합니다.

In [31]:
# 프롬프트 길이별 TTFT 비교
prompts = {
    "짧은 프롬프트": "안녕?",
    "중간 프롬프트": "파이썬 프로그래밍 언어의 역사, 특징, 장단점, 그리고 주요 활용 분야에 대해 설명해주세요.",
    "긴 프롬프트": "다음 내용을 분석해주세요: " + "파이썬은 범용 프로그래밍 언어입니다. " * 25,
}

for label, prompt in prompts.items():
    # 토큰 수 확인
    token_count = client.models.count_tokens(
        model=MODEL, contents=prompt
    ).total_tokens

    start = time.time()
    for i, chunk in enumerate(client.models.generate_content_stream(
        model=MODEL,
        contents=prompt,
        config={"max_output_tokens": 50, "thinking_config": {"thinking_budget": 0}},
    )):
        if i == 0:
            ttft = time.time() - start
            break

    print(f"[{label}] 입력 토큰: {token_count:>5} → TTFT: {ttft:.3f}초")

[짧은 프롬프트] 입력 토큰:     4 → TTFT: 0.298초
[중간 프롬프트] 입력 토큰:    27 → TTFT: 0.419초
[긴 프롬프트] 입력 토큰:   332 → TTFT: 1.670초


## 1.10 LCEL 체인 스트리밍

LangChain의 LCEL 체인(`prompt | model | parser`)에서도 `.stream()` 메서드를 사용할 수 있습니다.
체인의 마지막 단계가 스트리밍을 지원하면, 전체 체인이 스트리밍됩니다.

```python
chain = prompt_template | model | StrOutputParser()
for chunk in chain.stream({"question": "질문"}):
    print(chunk, end="")
```

아래 코드는 LCEL 체인 스트리밍을 시연합니다.

In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "한국어로 간결하게 답변하세요."),
    ("human", "{question}"),
])
parser = StrOutputParser()

chain = prompt_template | llm | parser

# 체인 스트리밍
for chunk in chain.stream({"question": "Docker와 Kubernetes의 차이는?"}):
    print(chunk, end="", flush=True)

print()

**Docker**는 컨테이너를 만들고 실행하는 **도구(플랫폼)**입니다.

**Kubernetes**는 이러한 컨테이너들을 대규모로 관리하고 배포하는 **오케스트레이션(관리) 시스템**입니다.

간단히 말해:
*   **Docker**: 개별 컨테이너를 만드는 데 사용
*   **Kubernetes**: 만들어진 컨테이너들을 수백, 수천 개 단위로 묶어 관리하고 자동화하는 데 사용

**비유:**
*   **Docker**: 자동차(컨테이너)를 만드는 공장
*   **Kubernetes**: 만들어진 자동차들을 관리하고, 주차시키고, 사고 나면 다른 차로 교체해주고, 교통 흐름을 제어하는 교통 시스템


### StrOutputParser의 스트리밍 동작

`StrOutputParser`는 `AIMessageChunk`에서 `.content`를 추출하여 문자열 청크로 변환합니다.
parser를 사용하면 `chunk.content` 대신 바로 문자열이 나옵니다.

아래 코드는 parser 유무에 따른 청크 타입 차이를 보여줍니다.

In [33]:
# parser 유무에 따른 청크 타입 비교
question = {"question": "HTTP란?"}

# parser 없이 (model까지만)
chain_no_parser = prompt_template | llm
chunks_no_parser = list(chain_no_parser.stream(question))

# parser 있으면
chain_with_parser = prompt_template | llm | parser
chunks_with_parser = list(chain_with_parser.stream(question))

print("=== parser 없이 ===")
print(f"청크 타입: {type(chunks_no_parser[0]).__name__}")
print(f"첫 청크: {chunks_no_parser[0].content[:50]}")

print("\n=== parser 있으면 ===")
print(f"청크 타입: {type(chunks_with_parser[0]).__name__}")
print(f"첫 청크: {chunks_with_parser[0][:50]}")

=== parser 없이 ===
청크 타입: AIMessageChunk
첫 청크: **HTTP(Hypertext Transfer Protocol

=== parser 있으면 ===
청크 타입: TextAccessor
첫 청크: **HTTP (Hyper


## 1.11 스트리밍과 생성 파라미터

노트북 5에서 배운 생성 파라미터(temperature, max_output_tokens, stop_sequences 등)는
스트리밍에서도 동일하게 작동합니다.

아래 코드는 `max_output_tokens`를 설정한 스트리밍에서 `finish_reason`을 확인합니다.

In [34]:
# max_output_tokens + 스트리밍
prompt = "1부터 100까지의 소수를 모두 나열해줘"

full_text = ""
last_chunk = None

for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"max_output_tokens": 30, "thinking_config": {"thinking_budget": 0}},
):
    full_text += chunk.text
    last_chunk = chunk
    print(chunk.text, end="", flush=True)

print(f"\n\nfinish_reason: {last_chunk.candidates[0].finish_reason}")
print(f"출력 토큰: {last_chunk.usage_metadata.candidates_token_count}")
print("(MAX_TOKENS면 응답이 잘린 것)")

1부터 100까지의 소수는 다음과 같습니다:

2, 3, 5, 7, 11, 

finish_reason: FinishReason.MAX_TOKENS
출력 토큰: 28
(MAX_TOKENS면 응답이 잘린 것)


### stop_sequences + 스트리밍

`stop_sequences`도 스트리밍에서 동일하게 작동합니다.
지정된 문자열이 생성되는 순간 스트리밍이 즉시 중단됩니다.

In [36]:
# stop_sequences + 스트리밍
prompt = "좋은 프로그래밍 습관 5가지를 번호를 매겨 알려줘"

print("=== stop 없이 ===")
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"max_output_tokens": 200, "thinking_config": {"thinking_budget": 0}},
):
    print(chunk.text, end="", flush=True)

print("\n\n=== stop=['4.', '4)'] (3개까지만) ===")
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={
        "stop_sequences": ["4.", "4)", "4"],
        "max_output_tokens": 2048,
        "thinking_config": {"thinking_budget": 0},
    },
):
    print(chunk.text, end="", flush=True)

print()

=== stop 없이 ===
## 좋은 프로그래밍 습관 5가지

좋은 프로그래밍 습관은 코드를 더 읽기 쉽고, 유지보수하기 쉬우며, 버그를 줄이는 데 크게 기여합니다. 다음은 강력히 권장되는 5가지 습관입니다.

1.  **코드를 자주 커밋하고, 의미 있는 커밋 메시지를 작성하기:**
    *   **자주 커밋:** 작업 중간중간, 기능 하나가 완성될 때마다 또는 중요한 변경사항이 있을 때마다 커밋하여 코드의 스냅샷을 만듭니다. 이렇게 하면 문제가 발생했을 때 특정 시점으로 쉽게 되돌아갈 수 있고, 작업 손실을 방지할 수 있습니다.
    *   **의미 있는 메시지:** 커밋 메시지는 해당 커밋에서 무엇을 변경했는지, 왜 변경했는지, 어떤 문제를 해결했는지를 명확하게 설명해야 합니다. "수정", "업

=== stop=['4.', '4)'] (3개까지만) ===
좋은 프로그래밍 습관 5가지는 다음과 같습니다.

1. **코드 정리 및 주석 달기:** 코드를 작성할 때는 항상 깔끔하고 읽기 쉽게 정리해야 합니다. 변수명은 의미를 명확히 나타내고, 함수는 하나의 기능만 수행하도록 작성하며, 적절한 들여쓰기를 사용해야 합니다. 또한, 코드의 복잡하거나 중요한 부분에는 주석을 달아 다른 사람이 이해하기 쉽도록 해야 합니다. 주석은 왜 그렇게 코드를 작성했는지, 무엇을 하는 코드인지 등을 설명해야 합니다.

2. **테스트 코드 작성:** 코드를 작성하는 것만큼 중요한 것이 코드가 제대로 동작하는지 확인하는 것입니다. 이를 위해 테스트 코드를 작성하는 습관을 들이세요. 테스트 코드는 작성한 코드의 각 기능이 의도한 대로 작동하는지 검증하고, 나중에 코드를 변경했을 때 발생할 수 있는 잠재적인 문제를 미리 발견하는 데 도움을 줍니다. 단위 테스트, 통합 테스트 등 다양한 종류의 테스트를 활용할 수 있습니다.

3. **버전 관리 시스템 사용:** Git과 같은 버전 관리 시스템을 사용하는 것은 현대 프로그래밍에서 필수적인 습관입니다. 버전 관리 시스템은 코드의 변경 

## 1.12 비동기 스트리밍 (astream)

웹 서버(FastAPI, Streamlit 등)에서는 비동기(async) 스트리밍이 유용합니다.
동기 스트리밍은 응답을 기다리는 동안 스레드를 점유하지만,
비동기 스트리밍은 다른 요청을 동시에 처리할 수 있습니다.

LangChain은 `.astream()` 메서드를, google-genai는 `client.aio.models.generate_content_stream()`을 제공합니다.

아래 코드는 LangChain의 비동기 스트리밍 예시입니다.

In [ ]:
# LangChain 비동기 스트리밍
import asyncio

async def stream_async():
    prompt = "사이버 보안의 중요성을 간단히 설명해줘"
    async for chunk in llm.astream(prompt):
        print(chunk.content, end="", flush=True)
    print()

await stream_async()

In [ ]:
# google-genai 비동기 스트리밍
async def stream_genai_async():
    prompt = "사이버 보안의 중요성을 간단히 설명해줘"
    async_client = genai.Client(api_key=GEMINI_API_KEY)

    async for chunk in async_client.aio.models.generate_content_stream(
        model=MODEL,
        contents=prompt,
        config={"thinking_config": {"thinking_budget": 0}},
    ):
        print(chunk.text, end="", flush=True)
    print()

await stream_genai_async()

> 동기 vs 비동기 스트리밍 비교
>
> | 항목 | 동기 (stream) | 비동기 (astream) |
> |------|-------------|----------------|
> | 사용 환경 | 스크립트, 노트북 | 웹 서버, 비동기 앱 |
> | 스레드 점유 | 응답 중 스레드 블로킹 | 대기 중 다른 작업 가능 |
> | 코드 복잡도 | `for chunk in ...` | `async for chunk in ...` |
> | LangChain 메서드 | `.stream()` | `.astream()` |
> | google-genai 메서드 | `generate_content_stream()` | `aio.models.generate_content_stream()` |
> | 성능 차이 | 단일 요청에서는 동일 | 동시 요청이 많을 때 이점 |

---

### 생각해보기

1. 스트리밍은 총 응답 시간을 줄이지 않는데, 왜 대부분의 챗봇 서비스가 스트리밍을 기본으로 사용할까요?
2. TTFT가 5초인 서비스와 TTFT가 0.5초인 서비스가 있다면, 사용자 이탈률에 어떤 차이가 있을까요?
3. 스트리밍 중 네트워크가 끊기면 어떤 일이 발생할까요? 이를 어떻게 처리해야 할까요?

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 6-1: google-genai 스트리밍 구현

google-genai SDK로 스트리밍 응답을 구현하세요.

**요구사항**
1. 프롬프트: "대한민국의 사계절 특징을 각각 설명해줘"
2. `generate_content_stream()`으로 스트리밍 호출
3. 각 청크의 텍스트를 실시간으로 출력 (`end=""`, `flush=True`)
4. 총 청크 수를 마지막에 출력

In [ ]:
# TODO: google-genai 스트리밍 구현
prompt = "대한민국의 사계절 특징을 각각 설명해줘"

# ---------- 정답 ----------
chunk_count = 0
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
):
    print(chunk.text, end="", flush=True)
    chunk_count += 1

print(f"\n\n총 청크 수: {chunk_count}")

print("TODO를 완성해주세요")

### 실습 6-2: LangChain 스트리밍 + 청크 확인

LangChain으로 스트리밍을 구현하고, 첫 번째 청크와 마지막 청크의 구조를 분석하세요.

**요구사항**
1. 프롬프트: "좋은 코드 리뷰의 원칙 3가지를 알려줘"
2. `llm.stream()`으로 스트리밍 호출
3. 모든 청크를 리스트에 저장
4. 첫 번째 청크의 `type`, `content`, `id` 출력
5. 마지막 청크의 `usage_metadata` 출력

In [ ]:
# TODO: LangChain 스트리밍 + 청크 분석
prompt = "좋은 코드 리뷰의 원칙 3가지를 알려줘"

# ---------- 정답 ----------
all_chunks = []
for chunk in llm.stream(prompt):
    all_chunks.append(chunk)
    print(chunk.content, end="", flush=True)

print(f"\n\n--- 청크 분석 ---")
print(f"총 청크 수: {len(all_chunks)}")

first = all_chunks[0]
print(f"\n첫 번째 청크:")
print(f"  type: {type(first).__name__}")
print(f"  content: {first.content[:50]}")
print(f"  id: {first.id}")

last = all_chunks[-1]
print(f"\n마지막 청크:")
print(f"  usage_metadata: {last.usage_metadata}")

print("TODO를 완성해주세요")

### 실습 6-3: TTFT 측정

스트리밍의 TTFT(Time To First Token)를 측정하세요.

**요구사항**
1. 프롬프트: "클라우드 네이티브 아키텍처란 무엇인가?"
2. `time.time()`으로 시작 시간 기록
3. 첫 번째 청크 도착 시 TTFT 계산
4. 전체 완료 시 Total Time 계산
5. TTFT, Total Time, TTFT 비율(%)을 출력

In [ ]:
# TODO: TTFT 측정
prompt = "클라우드 네이티브 아키텍처란 무엇인가?"

# ---------- 정답 ----------
start = time.time()
ttft = None

for i, chunk in enumerate(client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
)):
    if i == 0:
        ttft = time.time() - start
    print(chunk.text, end="", flush=True)

total = time.time() - start
print(f"\n\nTTFT: {ttft:.3f}초")
print(f"Total: {total:.3f}초")
print(f"TTFT 비율: {ttft/total*100:.1f}%")

print("TODO를 완성해주세요")

### 실습 6-4: 스트리밍 텍스트 누적 + 토큰 사용량

스트리밍 중 텍스트를 누적하고, 완료 후 토큰 사용량을 확인하세요.

**요구사항**
1. 프롬프트: "Git의 핵심 명령어 5가지와 각각의 용도를 설명해줘"
2. 스트리밍 중 텍스트를 변수에 누적 (`full_text += chunk.text`)
3. 마지막 청크에서 `usage_metadata` 저장
4. 완료 후 전체 텍스트 길이, 입력/출력/총 토큰 수를 출력

In [ ]:
# TODO: 텍스트 누적 + 토큰 사용량
prompt = "Git의 핵심 명령어 5가지와 각각의 용도를 설명해줘"

# ---------- 정답 ----------
full_text = ""
last_chunk = None

for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
):
    full_text += chunk.text
    last_chunk = chunk
    print(chunk.text, end="", flush=True)

usage = last_chunk.usage_metadata
print(f"\n\n--- 결과 ---")
print(f"전체 텍스트 길이: {len(full_text)}자")
print(f"입력 토큰: {usage.prompt_token_count}")
print(f"출력 토큰: {usage.candidates_token_count}")
print(f"총 토큰: {usage.total_token_count}")

print("TODO를 완성해주세요")

### 실습 6-5: LCEL 체인 스트리밍

LCEL 체인을 구성하고 스트리밍을 적용하세요.

**요구사항**
1. ChatPromptTemplate: system="전문가처럼 답변하세요", human="{topic}에 대해 설명해줘"
2. StrOutputParser 추가
3. `chain.stream({"topic": "마이크로서비스"})`로 스트리밍 실행
4. 각 청크의 타입을 확인 (str인지 확인)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO: LCEL 체인 스트리밍

# ---------- 정답 ----------
prompt_tpl = ChatPromptTemplate.from_messages([
    ("system", "전문가처럼 답변하세요"),
    ("human", "{topic}에 대해 설명해줘"),
])

chain = prompt_tpl | llm | StrOutputParser()

chunk_count = 0
for chunk in chain.stream({"topic": "마이크로서비스"}):
    print(chunk, end="", flush=True)
    chunk_count += 1
    if chunk_count == 1:
        print(f"\n(첫 청크 타입: {type(chunk).__name__})\n", end="")

print(f"\n\n총 청크 수: {chunk_count}")

print("TODO를 완성해주세요")

### 실습 6-6: 스트리밍 + 생성 파라미터

스트리밍에 `max_output_tokens`와 `stop_sequences`를 적용하세요.

**요구사항**
1. 프롬프트: "프로그래밍 언어를 10가지 번호를 매겨 나열해줘"
2. 호출 A: `max_output_tokens=50`으로 스트리밍 → finish_reason 확인
3. 호출 B: `stop_sequences=["6.", "6)"]`으로 스트리밍 → 5개까지만 출력
4. 두 결과를 비교 출력

In [ ]:
# TODO: 스트리밍 + 생성 파라미터
prompt = "프로그래밍 언어를 10가지 번호를 매겨 나열해줘"

# ---------- 정답 ----------
# A: max_output_tokens로 제한
print("=== max_output_tokens=50 ===")
last_chunk = None
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={"max_output_tokens": 50, "thinking_config": {"thinking_budget": 0}},
):
    print(chunk.text, end="", flush=True)
    last_chunk = chunk
print(f"\nfinish_reason: {last_chunk.candidates[0].finish_reason}")

# B: stop_sequences로 제한
print("\n=== stop_sequences=['6.', '6)'] ===")
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents=prompt,
    config={
        "stop_sequences": ["6.", "6)"],
        "thinking_config": {"thinking_budget": 0},
    },
):
    print(chunk.text, end="", flush=True)
print()

print("TODO를 완성해주세요")

---

### 생각해보기

1. 실습 6-4에서 확인한 토큰 사용량은 비스트리밍으로 같은 질문을 했을 때와 어떻게 다를까요? 입력 토큰은 동일할까요?
2. LCEL 체인에서 `StrOutputParser`를 빼면 청크 타입이 어떻게 바뀌나요? 실시간 출력 코드는 어떻게 수정해야 할까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 6-1: 스트리밍 vs 비스트리밍 성능 비교 실험 (난이도: ★★☆)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

스트리밍과 비스트리밍의 성능을 체계적으로 비교하는 실험을 설계하고 실행하세요.

**실험 설계**
1. 3가지 서로 다른 길이의 프롬프트 준비 (짧은/중간/긴)
2. 각 프롬프트에 대해 비스트리밍과 스트리밍 호출 실행
3. 측정 항목: TTFT, Total Time, 출력 토큰 수
4. 각 조합을 3회 반복하여 평균 계산

**분석 포인트**
- 프롬프트 길이에 따라 TTFT가 어떻게 변하는가?
- 비스트리밍 Total Time과 스트리밍 Total Time은 실제로 비슷한가?
- 어떤 상황에서 스트리밍의 UX 이점이 가장 큰가?

**결과 정리**
- 표 형태로 결과를 정리
- 결론과 권장사항을 작성

In [ ]:
# 1단계: 프롬프트 준비 + 측정 함수 구현
# 여기에 코드를 작성하세요

In [ ]:
# 2단계: 실험 실행 (3회 반복)
# 여기에 코드를 작성하세요

In [ ]:
# 3단계: 결과 분석 + 표 정리 + 결론
# 여기에 코드를 작성하세요

---

### 생각해보기

1. 실험 결과에서 스트리밍의 Total Time이 비스트리밍보다 약간 더 길 수 있습니다. 왜 그럴까요? 청크 전송의 오버헤드와 관련이 있을까요?
2. 실시간 채팅 서비스를 만든다면, 스트리밍과 비스트리밍 중 어떤 것을 기본으로 선택하겠습니까? 그 이유는?
3. 스트리밍 응답을 프론트엔드에 전달할 때 SSE(Server-Sent Events) 외에 WebSocket을 사용할 수도 있습니다. 두 방식의 장단점을 비교해보세요.